In [ ]:
import os
import urllib.request

os.makedirs("models", exist_ok=True)

if not os.path.exists("models/lid.176.ftz"):
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz",
        "models/lid.176.ftz",
    )

In [2]:
import fasttext

modelo = fasttext.load_model("models/lid.176.ftz")

In [ ]:
def detectar_idioma(texto):
    texto = texto.replace("\n", " ") + "\n"  # fastText necesita un solo salto de línea, al final
    resultado = modelo.f.predict(texto, 1, 0.0, "strict")  # devuelve [(probabilidad, "__label__es")]
    etiqueta = resultado[0][1]
    return etiqueta.replace("__label__", "")

In [4]:
print(detectar_idioma("hola como estas"))
print(detectar_idioma("olá, tudo bem? A comida estava ótima"))
print(detectar_idioma("The food was great but the wait was long"))

es
pt
en


In [5]:
import pandas as pd

archivos = [
    ("reseñas/resenas_ingles.csv", "Review Text", "en"),
    ("reseñas/resenas_espanol_trad.csv", "Review Text_es", "es"),
    ("reseñas/resenas_portugues_trad.csv", "Review Text_pt-BR", "pt"),
]

for archivo, columna, idioma_esperado in archivos:
    df = pd.read_csv(archivo)
    df["idioma_detectado"] = df[columna].apply(detectar_idioma)
    aciertos = (df["idioma_detectado"] == idioma_esperado).mean()
    print(f"{archivo}: {aciertos:.2%} de aciertos")

reseñas/resenas_ingles.csv: 100.00% de aciertos
reseñas/resenas_espanol_trad.csv: 100.00% de aciertos
reseñas/resenas_portugues_trad.csv: 99.95% de aciertos
